# 43 · Residual Flow and Local Jacobian Model

Notebook 42 suggested that residual behaves more like a **predictable deformation field** than like a shared aligned subspace.

This notebook refines that view by treating residual as a **flow along a continuous condition axis** rather than a single low→high jump.

## Core idea

Instead of only testing

\[
r_{\text{high}} \approx f(r_{\text{low}}, X),
\]

we ask whether residual evolves according to a local flow law

\[
\frac{dr}{dc} \approx g(r, X),
\]

where:
- \(c\) is a continuous condition coordinate
- \(r\) is the residual
- \(X\) is the local feature vector

## Main experiments

1. **Condition-axis parameterization**
2. **Finite-difference flow estimation**
3. **Local Jacobian modeling**
4. **Flow prediction quality**
5. **Flow integration test**
6. **Path-dependence / composition error**
7. **Residual vector-field view**

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Base data

In [ ]:
N = 2200
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
zeta_gaps_full = np.diff(t)

poisson_gaps_full = rng.exponential(scale=1.0, size=len(zeta_gaps_full))

def sample_gue_bulk_spacings(matrix_size=140, n_matrices=70, bulk_fraction=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    all_gaps = []
    for _ in range(n_matrices):
        real = rng.normal(size=(matrix_size, matrix_size))
        imag = rng.normal(size=(matrix_size, matrix_size))
        A = real + 1j * imag
        H = (A + A.conj().T) / 2.0
        eigvals = np.linalg.eigvalsh(H)
        eigvals = np.sort(eigvals)
        n = len(eigvals)
        keep = int(n * bulk_fraction)
        start = (n - keep) // 2
        stop = start + keep
        bulk_vals = eigvals[start:stop]
        all_gaps.extend(np.diff(bulk_vals).tolist())
    return np.array(all_gaps)

gue_gaps_full = sample_gue_bulk_spacings(matrix_size=140, n_matrices=70, bulk_fraction=0.5, rng=rng)

## Feature builders

In [ ]:
def extract_windows(gaps, k=5):
    return np.array([gaps[i:i+k] for i in range(len(gaps) - k + 1)])

def normalize_window(window):
    w = np.array(window, dtype=float)
    return w / w.mean()

def unevenness(window):
    return float(np.max(window) - np.min(window))

def reversal_symmetry_score(window):
    return float(np.mean(np.abs(window - window[::-1])))

def window_entropy(window):
    eps = 1e-12
    p = window / np.sum(window)
    return float(-np.sum(p * np.log(p + eps)))

def ratio_mean(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.mean(r))

def ratio_std(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.std(r))

def build_windows_and_features(gaps, feature_family="minimal", k=5):
    windows = extract_windows(gaps, k=k)
    windows_n = np.array([normalize_window(w) for w in windows])

    stats = {
        "entropy": np.array([window_entropy(w) for w in windows_n]),
        "unevenness": np.array([unevenness(w) for w in windows_n]),
        "ratio_mean": np.array([ratio_mean(w) for w in windows_n]),
        "symmetry": np.array([reversal_symmetry_score(w) for w in windows_n]),
    }

    if feature_family == "minimal":
        X = np.array([
            [stats["entropy"][i], stats["unevenness"][i], stats["ratio_mean"][i]]
            for i in range(len(windows_n))
        ], dtype=float)
    elif feature_family == "full":
        X = np.array([
            [stats["entropy"][i], stats["unevenness"][i], stats["symmetry"][i], stats["ratio_mean"][i], ratio_std(windows_n[i])]
            for i in range(len(windows_n))
        ], dtype=float)
    elif feature_family == "raw_window":
        X = windows_n.copy()
    else:
        raise ValueError(feature_family)

    return windows_n, stats, X

def make_contextual_features(X):
    prev_X = np.vstack([X[0], X[:-1]])
    next_X = np.vstack([X[1:], X[-1]])
    delta_prev = X - prev_X
    delta_next = next_X - X
    curv = next_X - 2 * X + prev_X
    return np.hstack([X, delta_prev, delta_next, curv])

def standardize_train_test(X_train, X_test):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std == 0] = 1.0
    return (X_train - mean) / std, (X_test - mean) / std, mean, std

def rank_normalize(x):
    order = np.argsort(x)
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(len(x), dtype=float)
    if len(x) <= 1:
        return np.zeros_like(ranks)
    return ranks / (len(x) - 1)

## Boundary models

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -40, 40)))

def fit_logistic_regression(X, y, lr=0.1, n_steps=2500, reg=1e-4):
    Xb = np.column_stack([np.ones(len(X)), X])
    w = np.zeros(Xb.shape[1])
    for _ in range(n_steps):
        p = sigmoid(Xb @ w)
        grad = Xb.T @ (p - y) / len(X)
        grad[1:] += reg * w[1:]
        w -= lr * grad
    return w

def decision_function_logistic(X, w):
    Xb = np.column_stack([np.ones(len(X)), X])
    return Xb @ w

def choose_prototypes(X, n_proto=20):
    idx = np.linspace(0, len(X) - 1, n_proto).astype(int)
    return X[idx]

def estimate_rbf_gamma(X):
    D = np.linalg.norm(X[:, None, :] - X[None, :, :], axis=2)
    pos = D[D > 0]
    med = np.median(pos) if len(pos) else 1.0
    if med <= 0:
        med = 1.0
    return 1.0 / (2.0 * med * med)

def rbf_features(X, prototypes, gamma):
    D2 = np.sum((X[:, None, :] - prototypes[None, :, :]) ** 2, axis=2)
    return np.exp(-gamma * D2)

def fit_linear_boundary(X0_train, X1_train):
    m = min(len(X0_train), len(X1_train))
    X0_train = X0_train[:m]
    X1_train = X1_train[:m]
    X_train = np.vstack([X0_train, X1_train])
    y_train = np.array([0] * len(X0_train) + [1] * len(X1_train))
    Xtr, _, mean, std = standardize_train_test(X_train, X_train)
    w = fit_logistic_regression(Xtr, y_train, lr=0.12, n_steps=2400, reg=1e-4)
    return {"kind": "linear", "mean": mean, "std": std, "w": w}

def fit_rbf_boundary(X0_train, X1_train, n_proto=20):
    m = min(len(X0_train), len(X1_train))
    X0_train = X0_train[:m]
    X1_train = X1_train[:m]
    X_train = np.vstack([X0_train, X1_train])
    y_train = np.array([0] * len(X0_train) + [1] * len(X1_train))
    Xtr, _, mean, std = standardize_train_test(X_train, X_train)
    prototypes = choose_prototypes(Xtr, n_proto=min(n_proto, len(Xtr)))
    gamma = estimate_rbf_gamma(Xtr)
    R_train = rbf_features(Xtr, prototypes, gamma)
    w = fit_logistic_regression(R_train, y_train, lr=0.15, n_steps=3500, reg=1e-4)
    return {"kind": "rbf", "mean": mean, "std": std, "prototypes": prototypes, "gamma": gamma, "w": w}

def score_model(model, X0_test, X1_test):
    m = min(len(X0_test), len(X1_test))
    X0_test = X0_test[:m]
    X1_test = X1_test[:m]
    X_test = np.vstack([X0_test, X1_test])
    y_test = np.array([0] * len(X0_test) + [1] * len(X1_test))
    Xte = (X_test - model["mean"]) / model["std"]

    if model["kind"] == "linear":
        scores = decision_function_logistic(Xte, model["w"])
    else:
        R_test = rbf_features(Xte, model["prototypes"], model["gamma"])
        scores = decision_function_logistic(R_test, model["w"])

    probs = sigmoid(scores)
    return y_test, scores, probs, X_test

## Flow and Jacobian helpers

In [ ]:
def fit_linear_regression(X, y, reg=1e-4):
    Xb = np.column_stack([np.ones(len(X)), X])
    I = np.eye(Xb.shape[1]); I[0,0] = 0.0
    beta = np.linalg.solve(Xb.T @ Xb + reg * I, Xb.T @ y)
    return beta

def predict_linear_regression(X, beta):
    Xb = np.column_stack([np.ones(len(X)), X])
    return Xb @ beta

def r2_score(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    if ss_tot <= 1e-12:
        return 0.0
    return float(1.0 - ss_res / ss_tot)

def correlation(x, y):
    x = np.asarray(x); y = np.asarray(y)
    if np.std(x) <= 1e-12 or np.std(y) <= 1e-12:
        return 0.0
    return float(np.corrcoef(x, y)[0, 1])

def fit_flow_linear(r, X, flow):
    Z = np.column_stack([r, X])
    Zs, _, mean, std = standardize_train_test(Z, Z)
    beta = fit_linear_regression(Zs, flow, reg=1e-3)
    return {"kind": "linear", "mean": mean, "std": std, "beta": beta}

def fit_flow_contextual(r, X, flow):
    Xc = make_contextual_features(X)
    Z = np.column_stack([r, Xc])
    Zs, _, mean, std = standardize_train_test(Z, Z)
    beta = fit_linear_regression(Zs, flow, reg=1e-3)
    return {"kind": "contextual", "mean": mean, "std": std, "beta": beta}

def fit_flow_nonlinear(r, X, flow, n_proto=20):
    Xc = make_contextual_features(X)
    Z = np.column_stack([r, Xc])
    Zs, _, mean, std = standardize_train_test(Z, Z)
    prototypes = choose_prototypes(Zs, n_proto=min(n_proto, len(Zs)))
    gamma = estimate_rbf_gamma(Zs)
    R = rbf_features(Zs, prototypes, gamma)
    beta = fit_linear_regression(R, flow, reg=1e-3)
    return {"kind": "nonlinear", "mean": mean, "std": std, "prototypes": prototypes, "gamma": gamma, "beta": beta}

def predict_flow(model, r, X):
    if model["kind"] == "linear":
        Z = np.column_stack([r, X])
        Zs = (Z - model["mean"]) / model["std"]
        return predict_linear_regression(Zs, model["beta"])
    elif model["kind"] == "contextual":
        Xc = make_contextual_features(X)
        Z = np.column_stack([r, Xc])
        Zs = (Z - model["mean"]) / model["std"]
        return predict_linear_regression(Zs, model["beta"])
    else:
        Xc = make_contextual_features(X)
        Z = np.column_stack([r, Xc])
        Zs = (Z - model["mean"]) / model["std"]
        R = rbf_features(Zs, model["prototypes"], model["gamma"])
        return predict_linear_regression(R, model["beta"])

def finite_difference_flow(r_sorted, c_sorted):
    dc = np.diff(c_sorted)
    dr = np.diff(r_sorted)
    flow = dr / (dc + 1e-6)
    c_mid = 0.5 * (c_sorted[1:] + c_sorted[:-1])
    r_mid = 0.5 * (r_sorted[1:] + r_sorted[:-1])
    return flow, c_mid, r_mid

def local_jacobian_proxy(r_mid, X_mid, flow):
    Z = np.column_stack([r_mid, X_mid])
    Zs, _, _, _ = standardize_train_test(Z, Z)
    beta = fit_linear_regression(Zs, flow, reg=1e-3)
    return beta[1:]

def integrate_flow_path(c_train, flow_pred_train, c_target_sorted, r0):
    order = np.argsort(c_train)
    c_ref = c_train[order]
    f_ref = flow_pred_train[order]
    r_pred = np.zeros(len(c_target_sorted))
    r_pred[0] = r0
    for i in range(1, len(c_target_sorted)):
        c_prev = c_target_sorted[i-1]
        c_cur = c_target_sorted[i]
        dc = c_cur - c_prev
        f_local = np.interp(c_prev, c_ref, f_ref)
        r_pred[i] = r_pred[i-1] + f_local * dc
    return r_pred

def path_dependence_error(r_direct, r_via_mid):
    return float(np.mean(np.abs(r_direct - r_via_mid)))

## Forced residual extractor

In [ ]:
def extract_forced_residual(X0_low, X1_low, X0_high, X1_high, forcing_mode):
    train0 = np.vstack([X0_low, X0_high])
    train1 = np.vstack([X1_low, X1_high])

    if forcing_mode == "capacity_gap":
        core_model = fit_linear_boundary(train0, train1)
        spec_model = fit_rbf_boundary(train0, train1, n_proto=20)
        low = score_model(core_model, X0_low, X1_low), score_model(spec_model, X0_low, X1_low)
        high = score_model(core_model, X0_high, X1_high), score_model(spec_model, X0_high, X1_high)
        mixed = score_model(core_model, train0, train1), score_model(spec_model, train0, train1)
    elif forcing_mode == "feature_gap":
        ctx_train0, ctx_train1 = make_contextual_features(train0), make_contextual_features(train1)
        ctx_low0, ctx_low1 = make_contextual_features(X0_low), make_contextual_features(X1_low)
        ctx_high0, ctx_high1 = make_contextual_features(X0_high), make_contextual_features(X1_high)
        core_model = fit_rbf_boundary(train0, train1, n_proto=20)
        spec_model = fit_rbf_boundary(ctx_train0, ctx_train1, n_proto=20)
        low = score_model(core_model, X0_low, X1_low), score_model(spec_model, ctx_low0, ctx_low1)
        high = score_model(core_model, X0_high, X1_high), score_model(spec_model, ctx_high0, ctx_high1)
        mixed = score_model(core_model, train0, train1), score_model(spec_model, ctx_train0, ctx_train1)
    elif forcing_mode == "condition_gap":
        core_model = fit_rbf_boundary(train0, train1, n_proto=20)
        spec_model_low = fit_rbf_boundary(X0_low, X1_low, n_proto=20)
        spec_model_high = fit_rbf_boundary(X0_high, X1_high, n_proto=20)
        spec_model_mixed = fit_rbf_boundary(X0_high, X1_high, n_proto=20)
        low = score_model(core_model, X0_low, X1_low), score_model(spec_model_low, X0_low, X1_low)
        high = score_model(core_model, X0_high, X1_high), score_model(spec_model_high, X0_high, X1_high)
        mixed = score_model(core_model, train0, train1), score_model(spec_model_mixed, train0, train1)
    else:
        raise ValueError(forcing_mode)

    def pack(core_out, spec_out):
        y_true, s_core, p_core, X = core_out
        _, s_spec, _, _ = spec_out
        residual_raw = s_spec - s_core
        residual = residual_raw / (np.std(residual_raw) + 1e-6)
        return {"y_true": y_true, "X": X, "s_core": s_core, "p_core": p_core, "s_spec": s_spec, "residual_raw": residual_raw, "residual": residual}

    return {"low": pack(*low), "high": pack(*high), "mixed": pack(*mixed)}

## Experiment grid

In [ ]:
window_sizes = [3, 5, 7]
feature_family = "minimal"
sample_size = 100
height_block = (0, 400)

systems = ["entropy", "unevenness"]
tasks = {
    "zeta_vs_GUE": ("zeta", "gue"),
    "zeta_vs_Poisson": ("zeta", "poisson"),
}
forcing_modes = ["capacity_gap", "feature_gap", "condition_gap"]
flow_modes = ["linear", "contextual", "nonlinear"]

## Base gap slices

In [ ]:
start, stop = height_block
base_gaps = {
    "zeta": zeta_gaps_full[start:stop],
    "poisson": poisson_gaps_full[start:stop],
    "gue": gue_gaps_full[:max(stop - start + 300, 900)],
}

## Continuous condition features

In [ ]:
def get_rank_conditioned_features(gaps, system_name, k=5, feature_family="minimal", n=100):
    _, stats, X = build_windows_and_features(gaps, feature_family=feature_family, k=k)
    c = rank_normalize(stats[system_name])
    idx = np.argsort(c)
    idx = idx[:min(len(idx), n)]
    return X[idx], c[idx]

def split_low_mid_high(X, c, n=30):
    order = np.argsort(c)
    Xs = X[order]
    cs = c[order]
    m = min(n, len(Xs)//3)
    return {
        "low": (Xs[:m], cs[:m]),
        "mid": (Xs[len(Xs)//2 - m//2: len(Xs)//2 - m//2 + m], cs[len(Xs)//2 - m//2: len(Xs)//2 - m//2 + m]),
        "high": (Xs[-m:], cs[-m:]),
        "all": (Xs, cs),
    }

## Main residual-flow sweep

In [ ]:
results = []
vector_cache = {}
scatter_cache = {}

for system_name in systems:
    for task_name, (ens0, ens1) in tasks.items():
        for k in window_sizes:
            X0_all, c0_all = get_rank_conditioned_features(base_gaps[ens0], system_name, k=k, feature_family=feature_family, n=sample_size)
            X1_all, c1_all = get_rank_conditioned_features(base_gaps[ens1], system_name, k=k, feature_family=feature_family, n=sample_size)
            m = min(len(X0_all), len(X1_all), sample_size)
            if m < 45:
                continue

            X0_all, c0_all = X0_all[:m], c0_all[:m]
            X1_all, c1_all = X1_all[:m], c1_all[:m]

            split0 = split_low_mid_high(X0_all, c0_all, n=min(30, m//3))
            split1 = split_low_mid_high(X1_all, c1_all, n=min(30, m//3))

            X0_low, c0_low = split0["low"]
            X0_mid, c0_mid = split0["mid"]
            X0_high, c0_high = split0["high"]

            X1_low, c1_low = split1["low"]
            X1_mid, c1_mid = split1["mid"]
            X1_high, c1_high = split1["high"]

            for forcing_mode in forcing_modes:
                data_lm = extract_forced_residual(X0_low, X1_low, X0_mid, X1_mid, forcing_mode)
                data_mh = extract_forced_residual(X0_mid, X1_mid, X0_high, X1_high, forcing_mode)
                data_lh = extract_forced_residual(X0_low, X1_low, X0_high, X1_high, forcing_mode)

                r_lm = data_lm["mixed"]["residual"]
                X_lm = data_lm["mixed"]["X"]
                c_lm = np.linspace(0.0, 0.5, len(r_lm))

                r_mh = data_mh["mixed"]["residual"]
                X_mh = data_mh["mixed"]["X"]
                c_mh = np.linspace(0.5, 1.0, len(r_mh))

                r_lh = data_lh["mixed"]["residual"]
                X_lh = data_lh["mixed"]["X"]
                c_lh = np.linspace(0.0, 1.0, len(r_lh))

                order_lh = np.argsort(c_lh)
                r_lh_s = r_lh[order_lh]
                X_lh_s = X_lh[order_lh]
                c_lh_s = c_lh[order_lh]

                flow_lh, c_mid_lh, r_mid_lh = finite_difference_flow(r_lh_s, c_lh_s)
                X_mid_lh = 0.5 * (X_lh_s[1:] + X_lh_s[:-1])

                order_lm = np.argsort(c_lm)
                flow_lm, c_mid_lm, r_mid_lm = finite_difference_flow(r_lm[order_lm], c_lm[order_lm])
                X_mid_lm = 0.5 * (X_lm[order_lm][1:] + X_lm[order_lm][:-1])

                order_mh = np.argsort(c_mh)
                flow_mh, c_mid_mh, r_mid_mh = finite_difference_flow(r_mh[order_mh], c_mh[order_mh])
                X_mid_mh = 0.5 * (X_mh[order_mh][1:] + X_mh[order_mh][:-1])

                jac = local_jacobian_proxy(r_mid_lh, X_mid_lh, flow_lh)
                jac_norm = float(np.linalg.norm(jac))

                for flow_mode in flow_modes:
                    if flow_mode == "linear":
                        model = fit_flow_linear(r_mid_lh, X_mid_lh, flow_lh)
                        m_lm = fit_flow_linear(r_mid_lm, X_mid_lm, flow_lm)
                        m_mh = fit_flow_linear(r_mid_mh, X_mid_mh, flow_mh)
                    elif flow_mode == "contextual":
                        model = fit_flow_contextual(r_mid_lh, X_mid_lh, flow_lh)
                        m_lm = fit_flow_contextual(r_mid_lm, X_mid_lm, flow_lm)
                        m_mh = fit_flow_contextual(r_mid_mh, X_mid_mh, flow_mh)
                    else:
                        model = fit_flow_nonlinear(r_mid_lh, X_mid_lh, flow_lh, n_proto=20)
                        m_lm = fit_flow_nonlinear(r_mid_lm, X_mid_lm, flow_lm, n_proto=20)
                        m_mh = fit_flow_nonlinear(r_mid_mh, X_mid_mh, flow_mh, n_proto=20)

                    flow_pred = predict_flow(model, r_mid_lh, X_mid_lh)
                    flow_r2 = r2_score(flow_lh, flow_pred)
                    flow_corr = correlation(flow_lh, flow_pred)

                    r_int = integrate_flow_path(c_mid_lh, flow_pred, c_lh_s, r_lh_s[0])
                    integration_r2 = r2_score(r_lh_s, r_int)
                    integration_mae = float(np.mean(np.abs(r_lh_s - r_int)))

                    pred_lm = predict_flow(m_lm, r_mid_lm, X_mid_lm)
                    pred_mh = predict_flow(m_mh, r_mid_mh, X_mid_mh)

                    r_lm_int = integrate_flow_path(c_mid_lm, pred_lm, np.sort(c_lm), np.sort(r_lm)[0])
                    r_mh_int = integrate_flow_path(c_mid_mh, pred_mh, np.sort(c_mh), np.sort(r_mh)[0])

                    bridge_direct = np.interp(np.linspace(0, 1, 50), c_lh_s, r_int)
                    bridge_piecewise = np.concatenate([
                        np.interp(np.linspace(0, 0.5, 25), np.sort(c_lm), r_lm_int),
                        np.interp(np.linspace(0.5, 1.0, 25), np.sort(c_mh), r_mh_int),
                    ])
                    curl_proxy = path_dependence_error(bridge_direct, bridge_piecewise)

                    results.append({
                        "system": system_name,
                        "task": task_name,
                        "k": k,
                        "forcing_mode": forcing_mode,
                        "flow_mode": flow_mode,
                        "flow_r2": flow_r2,
                        "flow_corr": flow_corr,
                        "integration_r2": integration_r2,
                        "integration_mae": integration_mae,
                        "curl_proxy": curl_proxy,
                        "jacobian_norm": jac_norm,
                        "mean_flow_mag": float(np.mean(np.abs(flow_lh))),
                    })

                    if system_name == "entropy" and task_name == "zeta_vs_GUE" and forcing_mode == "condition_gap" and flow_mode == "nonlinear" and k == 5:
                        scatter_cache["example"] = {"c_sorted": c_lh_s, "r_sorted": r_lh_s, "r_int": r_int}
                        vector_cache["example"] = {"c_mid": c_mid_lh, "r_mid": r_mid_lh, "flow_pred": flow_pred}

len(results)

## Plot 1 · Residual vs condition

In [ ]:
ex = scatter_cache["example"]
fig, ax = plt.subplots(figsize=(6.8, 4.8))
ax.scatter(ex["c_sorted"], ex["r_sorted"], s=14, alpha=0.7, label="observed residual")
ax.plot(ex["c_sorted"], ex["r_int"], linewidth=2, label="integrated flow prediction")
ax.set_xlabel("condition coordinate c")
ax.set_ylabel("residual")
ax.set_title("Residual vs condition · condition_gap · nonlinear · entropy · zeta_vs_GUE · k=5")
ax.legend()
plt.tight_layout()
plt.show()

## Plot 2 · Flow prediction quality

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)

for ax, metric, title in zip(
    axes,
    ["flow_r2", "integration_r2"],
    ["Flow model $R^2$", "Integrated residual $R^2$"]
):
    for flow_mode in flow_modes:
        rows = [r for r in results if r["flow_mode"] == flow_mode and r["forcing_mode"] == "condition_gap" and r["task"] == "zeta_vs_GUE" and r["system"] == "entropy"]
        rows = sorted(rows, key=lambda x: x["k"])
        ax.plot([r["k"] for r in rows], [r[metric] for r in rows], marker="o", label=flow_mode)
    ax.set_title(title)
    ax.set_xlabel("window size k")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Plot 3 · Flow heatmaps

In [ ]:
def build_matrix(metric, system_name, forcing_mode):
    rows = [r for r in results if r["system"] == system_name and r["task"] == "zeta_vs_GUE" and r["forcing_mode"] == forcing_mode]
    return np.array([
        [next(r for r in rows if r["k"] == k and r["flow_mode"] == fm)[metric] for k in window_sizes]
        for fm in flow_modes
    ])

fig, axes = plt.subplots(3, 2, figsize=(10, 11), sharex=True, sharey=True)
for i, forcing_mode in enumerate(forcing_modes):
    for j, system_name in enumerate(["entropy", "unevenness"]):
        ax = axes[i, j]
        M = build_matrix("flow_r2", system_name, forcing_mode)
        im = ax.imshow(M, aspect="auto", origin="upper")
        ax.set_title(f"{forcing_mode} · {system_name}")
        ax.set_xticks(np.arange(len(window_sizes)), window_sizes)
        ax.set_yticks(np.arange(len(flow_modes)), flow_modes)
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85, label="flow $R^2$")
plt.tight_layout()
plt.show()

## Plot 4 · Path dependence / curl proxy

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)

for ax, system_name in zip(axes, ["entropy", "unevenness"]):
    for flow_mode in flow_modes:
        rows = [r for r in results if r["flow_mode"] == flow_mode and r["forcing_mode"] == "condition_gap" and r["task"] == "zeta_vs_GUE" and r["system"] == system_name]
        rows = sorted(rows, key=lambda x: x["k"])
        ax.plot([r["k"] for r in rows], [r["curl_proxy"] for r in rows], marker="o", label=flow_mode)
    ax.set_title(system_name)
    ax.set_xlabel("window size k")
    ax.legend(fontsize=8)

axes[0].set_ylabel("path dependence error")
plt.tight_layout()
plt.show()

## Plot 5 · Jacobian norm and flow magnitude

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=False)

for ax, metric, title in zip(
    axes,
    ["jacobian_norm", "mean_flow_mag"],
    ["Local Jacobian norm", "Mean |flow|"]
):
    for flow_mode in flow_modes:
        rows = [r for r in results if r["flow_mode"] == flow_mode and r["forcing_mode"] == "condition_gap" and r["task"] == "zeta_vs_GUE" and r["system"] == "entropy"]
        rows = sorted(rows, key=lambda x: x["k"])
        ax.plot([r["k"] for r in rows], [r[metric] for r in rows], marker="o", label=flow_mode)
    ax.set_title(title)
    ax.set_xlabel("window size k")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Plot 6 · Vector-field view

In [ ]:
vec = vector_cache["example"]
fig, ax = plt.subplots(figsize=(6.8, 4.8))
step = max(1, len(vec["c_mid"]) // 25)
ax.quiver(
    vec["c_mid"][::step],
    vec["r_mid"][::step],
    np.full(len(vec["c_mid"][::step]), 0.03),
    vec["flow_pred"][::step],
    angles="xy",
    scale_units="xy",
    scale=1.0,
    width=0.003,
)
ax.set_xlabel("condition coordinate c")
ax.set_ylabel("residual")
ax.set_title("Residual flow field · condition_gap · nonlinear · entropy · zeta_vs_GUE · k=5")
plt.tight_layout()
plt.show()

## Summary table

In [ ]:
summary_rows = []
for forcing_mode in forcing_modes:
    for system_name in systems:
        for task_name in tasks:
            for k in window_sizes:
                candidates = [r for r in results if r["forcing_mode"] == forcing_mode and r["system"] == system_name and r["task"] == task_name and r["k"] == k]
                best = max(candidates, key=lambda r: r["integration_r2"])
                if best["integration_r2"] > 0.15 and best["curl_proxy"] < 0.25:
                    verdict = "smooth deformation field"
                elif best["flow_r2"] > 0.05:
                    verdict = "path-dependent nonlinear flow"
                else:
                    verdict = "weak local flow"
                summary_rows.append({
                    "forcing_mode": forcing_mode,
                    "system": system_name,
                    "task": task_name,
                    "k": k,
                    "best_flow_mode": best["flow_mode"],
                    "flow_r2": best["flow_r2"],
                    "integration_r2": best["integration_r2"],
                    "integration_mae": best["integration_mae"],
                    "curl_proxy": best["curl_proxy"],
                    "jacobian_norm": best["jacobian_norm"],
                    "verdict": verdict,
                })
summary_rows

## Compact summary

In [ ]:
summary = {
    "window_sizes": window_sizes,
    "forcing_modes": forcing_modes,
    "flow_modes": flow_modes,
    "systems": systems,
    "tasks": list(tasks.keys()),
    "summary_rows": summary_rows,
}
summary

## Reading guide

- **Flow $R^2$** asks whether the local derivative \(dr/dc\) is predictable from \((r, X)\).
- **Integrated residual $R^2$** asks whether the learned flow reconstructs residual along the condition axis.
- **Path dependence / curl proxy** asks whether direct transport and piecewise transport disagree.
- **Jacobian norm** measures how strongly local features contribute to deformation.
- **Vector field** gives a visual representation of residual motion along the condition axis.

## Verdicts

- **smooth deformation field** → local flow is predictive and integrates reasonably well
- **path-dependent nonlinear flow** → local flow exists but depends on path / regime
- **weak local flow** → deformation behaves mostly like noise at this scale